# TP Data Integration & Applications — ST2DLDI
**Dataset:** Road traffic injury accidents — France 2024 (data.gouv.fr)  
**Architecture:** Medallion (Bronze → Silver → Gold)

## Part 1 — Data Profiling & Data Quality

### Data loading (Bronze layer)

In [1]:
import pandas as pd
import numpy as np

characteristics = pd.read_csv('data/caract-2024.csv',    sep=';', encoding='latin1', low_memory=False)
locations       = pd.read_csv('data/lieux-2024.csv',     sep=';', encoding='latin1', low_memory=False)
users           = pd.read_csv('data/usagers-2024.csv',   sep=';', encoding='latin1', low_memory=False)
vehicles        = pd.read_csv('data/vehicules-2024.csv', sep=';', encoding='latin1', low_memory=False)

tables = {
    'characteristics': characteristics,
    'locations':       locations,
    'users':           users,
    'vehicles':        vehicles,
}

for name, df in tables.items():
    print(f'  {name:<20} {len(df):>7} rows  {df.shape[1]} columns')

  characteristics        54402 rows  15 columns
  locations              70248 rows  18 columns
  users                 125187 rows  16 columns
  vehicles               92678 rows  11 columns


### Quick data exploration

In [2]:
#first rows of each table
characteristics.head(3)

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,atm,col,adr,lat,long
0,202400000001,25,3,2024,07:40,2,70,70285,1,1,5,1,D438,"47,56277000","6,75832000"
1,202400000002,20,3,2024,15:05,1,21,21054,2,3,7,6,HOTEL DIEU (RUE DE L'),"47,02109000","4,83755000"
2,202400000003,22,3,2024,19:30,2,15,15012,1,1,1,6,AllÃ©e des Tilleuls,"44,90238400","2,49641800"


In [3]:
locations.head(3)

,Num_Acc,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
0,202400000001,3,D438,0,NaN,2,2,0,1,1,260,2,NaN,7,1,0,1,90
1,202400000002,4,HOTEL DIEU (RUE DE L'),0,NaN,2,2,0,1,-1,-1,1,NaN,-1,9,0,1,30
2,202400000002,4,POTERNE (RUE),0,NaN,1,1,0,1,-1,-1,1,NaN,-1,9,0,1,30


In [4]:
users.head(3)

,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202400000001,203Â 988Â 581,155Â 781Â 758,A01,1,1,3,1,2003.0,2,1,-1,-1,-1,-1,-1
1,202400000001,203Â 988Â 582,155Â 781Â 759,B01,1,1,1,1,1997.0,4,1,-1,-1,-1,-1,-1
2,202400000002,203Â 988Â 579,155Â 781Â 757,A01,10,3,3,2,1927.0,5,0,-1,-1,3,3,1


In [5]:
vehicles.head(3)

,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
0,202400000001,155Â 781Â 758,A01,1,7,0,2,1,13,1,NaN
1,202400000001,155Â 781Â 759,B01,2,14,0,2,2,21,1,NaN
2,202400000002,155Â 781Â 757,A01,1,10,0,1,3,15,1,NaN


In [6]:
#descriptive statistics
characteristics.describe()

,Num_Acc,jour,mois,an,lum,agg,int,atm,col
count,5.440200e+04,54402.000000,54402.000000,54402.0,54402.000000,54402.000000,54402.000000,54402.000000,54402.000000
mean,2.024000e+11,15.665490,6.615749,2024.0,1.949708,1.625161,2.108195,1.677493,4.018069
std,1.570465e+04,8.737761,3.361701,0.0,1.488462,0.484086,2.042602,1.736956,1.984560
min,2.024000e+11,1.000000,1.000000,2024.0,1.000000,1.000000,1.000000,1.000000,-1.000000
25%,2.024000e+11,8.000000,4.000000,2024.0,1.000000,1.000000,1.000000,1.000000,3.000000
50%,2.024000e+11,16.000000,7.000000,2024.0,1.000000,2.000000,1.000000,1.000000,3.000000
75%,2.024000e+11,23.000000,10.000000,2024.0,3.000000,2.000000,2.000000,1.000000,6.000000
max,2.024001e+11,31.000000,12.000000,2024.0,5.000000,2.000000,9.000000,9.000000,7.000000


In [7]:
users.describe()

,Num_Acc,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,etatp
count,1.251870e+05,125187.000000,125187.000000,125187.000000,125187.000000,122608.000000,125187.000000,125187.000000,125187.000000,125187.000000,125187.000000,125187.000000
mean,2.024000e+11,2.100745,1.335554,2.524208,1.272696,1985.075411,3.069009,1.938788,0.843906,-0.800994,-0.223002,-0.814605
std,1.567431e+04,2.584540,0.610862,1.374424,0.559575,19.327486,2.762129,2.384028,2.910122,1.067398,1.276891,0.638087
min,2.024000e+11,-1.000000,1.000000,1.000000,-1.000000,1914.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,2.024000e+11,1.000000,1.000000,1.000000,1.000000,1971.000000,0.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000
50%,2.024000e+11,1.000000,1.000000,3.000000,1.000000,1988.000000,4.000000,1.000000,0.000000,-1.000000,0.000000,-1.000000
75%,2.024000e+11,2.000000,2.000000,4.000000,2.000000,2001.000000,5.000000,2.000000,0.000000,-1.000000,0.000000,-1.000000
max,2.024001e+11,10.000000,3.000000,4.000000,2.000000,2024.000000,9.000000,9.000000,9.000000,9.000000,9.000000,3.000000


### A. Table structure

In [8]:
for name, df in tables.items():
    print(f'--- {name.upper()} ---')
    for col, dtype in df.dtypes.items():
        print(f'  {col:<30} {str(dtype)}')
    print()

--- CHARACTERISTICS ---
  Num_Acc                        int64
  jour                           int64
  mois                           int64
  an                             int64
  hrmn                           object
  lum                            int64
  dep                            object
  com                            object
  agg                            int64
  int                            int64
  atm                            int64
  col                            int64
  adr                            object
  lat                            object
  long                           object

--- LOCATIONS ---
  Num_Acc                        int64
  catr                           int64
  voie                           object
  v1                             int64
  v2                             object
  circ                           int64
  nbv                            object
  vosp                           int64
  prof                           int64
  pr        

### A. Semantic meaning of columns

**Source:** "Description des bases de donnees annuelles des accidents corporels de la circulation routiere - Annees de 2005 a 2024" (ONISR, data.gouv.fr, resource id 8ef4c2a3-91a0-4d98-ae3a-989bde87b62a). The codes below are verified against this document (pages 4-13, full field list). Non-exhaustive list: a few key columns per table to demonstrate understanding of the schema.

**Characteristics table** (1 row = 1 accident)

| Column | Meaning |
|---|---|
| Num_Acc | Unique accident identifier |
| lum | Lighting conditions (1=daylight, 2=dusk/dawn, 3=night without public lighting, ...) |
| atm | Weather conditions (1=normal, 2=rain, 3=snow, ...) |
| col | Collision type (1=head-on, 2=rear-end, 3=side, ...) |

**Locations table** (road characteristics)

| Column | Meaning |
|---|---|
| catr | Road category (1=motorway, 2=national road, 3=departmental road, ...) |
| surf | Road surface condition (1=normal, 2=wet, 3=puddles, ...) |
| vma | Maximum authorized speed |

**Users table** (1 row = 1 person involved)

| Column | Meaning |
|---|---|
| catu | User category (1=driver, 2=passenger, 3=pedestrian) |
| grav | Injury severity (1=uninjured, 2=killed, 3=hospitalized injury, 4=slight injury) |
| secu1 | Safety equipment (1=seatbelt, 2=helmet, 3=child restraint, ...) |

**Vehicles table** (1 row = 1 vehicle involved)

| Column | Meaning |
|---|---|
| catv | Vehicle category (1=bicycle, 2=moped, 7=car, ...) |
| choc | Initial impact point (1=front, 2=front right, 3=front left, ...) |
| motor | Engine type (1=fossil fuel, 2=hybrid, 3=electric, ...) |

### B. Missing values

In [9]:
for name, df in tables.items():
    print(f'--- {name.upper()} ---')
    missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
    missing = missing[missing > 0]
    if missing.empty:
        print('  no missing values')
    else:
        for col, pct in missing.items():
            flag = ' CRITICAL' if pct > 20 else ''
            print(f'  {col:<30} {pct:.1f}%{flag}')
    print()

--- CHARACTERISTICS ---
  adr                            4.2%

--- LOCATIONS ---
  lartpc                         100.0% CRITICAL
  v2                             91.6% CRITICAL
  voie                           19.0%

--- USERS ---
  an_nais                        2.1%

--- VEHICLES ---
  occutc                         99.0% CRITICAL



### C. Consistency and validity

In [10]:
#split Metropole / DOM-TOM by department code
#(a latitude threshold is unreliable: Saint-Pierre-et-Miquelon, dep 975, has a latitude
# within the mainland range 41-52, and southern Corsica, dep 2B, can fall below 41)
DOMTOM_DEP = {'971', '972', '973', '974', '975', '976', '977', '978', '986', '987', '988'}
outside_mainland = characteristics['dep'].astype(str).isin(DOMTOM_DEP).sum()
print(f'Coordinates outside mainland France (DOM-TOM): {outside_mainland}')

#negative (placeholder) values
for col in ['grav', 'sexe', 'place']:
    neg = (users[col] < 0).sum()
    if neg > 0:
        print(f'users.{col}: {neg} negative values (-1 = not specified)')

#ages (derived from an_nais)
age = 2024 - users['an_nais']
negative_ages = (age < 0).sum()
outlier_ages = (age > 110).sum()
print(f'Negative ages: {negative_ages}')
print(f'Ages > 110 years (outliers): {outlier_ages}')

#duplicates
print()
for name, df in tables.items():
    dup = df.duplicated().sum()
    print(f'  {name:<20} {dup} duplicate(s)')

Coordinates outside mainland France (DOM-TOM): 3344
users.sexe: 2395 negative values (-1 = not specified)
users.place: 3 negative values (-1 = not specified)
Negative ages: 0
Ages > 110 years (outliers): 0



  characteristics      0 duplicate(s)
  locations            2 duplicate(s)
  users                0 duplicate(s)


  vehicles             0 duplicate(s)


### C. Categorical anomalies

In [11]:
#expected values per categorical column
expected_categories = {
    'grav':   [1, 2, 3, 4],
    'sexe':   [1, 2],
    'catu':   [1, 2, 3, 4],
    'trajet': [0, 1, 2, 3, 4, 5, 9],
}

for col, valid in expected_categories.items():
    unique_values = users[col].dropna().unique()
    unexpected = [v for v in unique_values if v not in valid]
    if unexpected:
        print(f'users.{col}: unexpected values -> {unexpected}')
    else:
        print(f'users.{col}: OK (values = {sorted(unique_values.tolist())})')

print()
#check lum and atm columns in characteristics
for col, valid in {'lum': [1,2,3,4,5], 'atm': [1,2,3,4,5,6,7,8,9]}.items():
    vals = characteristics[col].dropna().unique()
    unexpected = [v for v in vals if v not in valid]
    if unexpected:
        print(f'characteristics.{col}: unexpected values -> {unexpected}')
    else:
        print(f'characteristics.{col}: OK')

users.grav: OK (values = [1, 2, 3, 4])
users.sexe: unexpected values -> [np.int64(-1)]
users.catu: OK (values = [1, 2, 3])
users.trajet: unexpected values -> [np.int64(-1)]

characteristics.lum: OK
characteristics.atm: OK


### D. Quality summary

| Issue | Column | Action |
|---|---|---|
| 100% empty | locations.lartpc | Drop |
| 99% empty | vehicles.occutc | Drop |
| 91.6% empty | locations.v2 | Drop |
| 19% empty | locations.voie | Impute 'Unknown' |
| 4.2% empty | characteristics.adr | Impute 'Unknown' |
| Outside mainland France | dep | Split off DOM-TOM (by department code) |
| Placeholder -1 values | sexe, place, col | Replace with NaN |
| Duplicates | locations | Drop (2 rows) |

### D. Impact analysis

- Empty columns: unusable, must be excluded before any processing
- Placeholder -1 values in `grav` and `sexe`: bias averages and distributions
- Coordinates outside mainland France: would bias any geographic or map-based analysis
- Duplicates in `locations`: would inflate counts on certain roads

## Part 2A — Transformations (Silver layer)

In [12]:
#1. drop columns that are almost entirely empty
locations = locations.drop(columns=['lartpc', 'v2'])
vehicles  = vehicles.drop(columns=['occutc'])

#2. duplicates
locations = locations.drop_duplicates()

#3. convert GPS coordinates
characteristics['lat']  = pd.to_numeric(characteristics['lat'].str.replace(',', '.'),  errors='coerce')
characteristics['long'] = pd.to_numeric(characteristics['long'].str.replace(',', '.'), errors='coerce')

#4. split mainland France / DOM-TOM (by department code, more reliable than a latitude threshold)
DOMTOM_DEP = {'971', '972', '973', '974', '975', '976', '977', '978', '986', '987', '988'}
is_domtom             = characteristics['dep'].astype(str).isin(DOMTOM_DEP)
domtom                = characteristics[is_domtom]
characteristics_metro = characteristics[~is_domtom].copy()
print(f'DOM-TOM set aside: {len(domtom)} rows')
print(f'Mainland France kept: {len(characteristics_metro)} rows')

#5. replace -1 placeholders with NaN
for col in ['sexe', 'place', 'trajet', 'secu1', 'secu2', 'secu3', 'locp', 'etatp']:
    users[col] = users[col].replace(-1, np.nan)
characteristics_metro['col'] = characteristics_metro['col'].replace(-1, np.nan)

#6. impute missing text values
characteristics_metro['adr'] = characteristics_metro['adr'].fillna('Unknown')
locations['voie']            = locations['voie'].fillna('Unknown')

#7. date column
characteristics_metro['date'] = pd.to_datetime(
    characteristics_metro[['an', 'mois', 'jour']].rename(columns={'an': 'year', 'mois': 'month', 'jour': 'day'}),
    errors='coerce'
)

#8. time-of-day enrichment
characteristics_metro['hour'] = characteristics_metro['hrmn'].str[:2].astype(float, errors='ignore')

def time_of_day_category(h):
    if pd.isna(h):   return 'Unknown'
    if 6  <= h < 12: return 'Morning'
    if 12 <= h < 18: return 'Afternoon'
    if 18 <= h < 22: return 'Evening'
    return 'Night'

characteristics_metro['time_of_day'] = characteristics_metro['hour'].apply(time_of_day_category)

#9. max severity enrichment per accident
max_severity = users.groupby('Num_Acc')['grav'].max().reset_index()
max_severity.columns = ['Num_Acc', 'max_severity']
characteristics_metro = characteristics_metro.merge(max_severity, on='Num_Acc', how='left')

#save Silver files
characteristics_metro.to_csv('data/silver_caract.csv',    index=False)
locations.to_csv(             'data/silver_lieux.csv',     index=False)
users.to_csv(                 'data/silver_usagers.csv',   index=False)
vehicles.to_csv(               'data/silver_vehicules.csv', index=False)
print('[OK] Silver files saved')

DOM-TOM set aside: 3344 rows
Mainland France kept: 51058 rows


[OK] Silver files saved


## Part 2B — Modeling (Gold layer)

In [13]:
characteristics_s = pd.read_csv('data/silver_caract.csv',    low_memory=False)
locations_s       = pd.read_csv('data/silver_lieux.csv',     low_memory=False)
users_s           = pd.read_csv('data/silver_usagers.csv',   low_memory=False)
vehicles_s        = pd.read_csv('data/silver_vehicules.csv', low_memory=False)

#fact table
fact_accidents = characteristics_s[[
    'Num_Acc', 'date', 'time_of_day', 'max_severity',
    'atm', 'col', 'lum', 'agg', 'dep'
]].rename(columns={
    'atm': 'weather', 'col': 'collision_type',
    'lum': 'lighting', 'agg': 'urban_area', 'dep': 'department'
})

#location dimension
dim_location = locations_s[[
    'Num_Acc', 'catr', 'surf', 'infra', 'situ', 'vma', 'prof', 'plan'
]].rename(columns={
    'catr': 'road_type', 'surf': 'road_surface', 'infra': 'infrastructure',
    'situ': 'situation', 'vma': 'speed_limit', 'prof': 'road_profile', 'plan': 'road_layout'
})

#vehicle dimension
dim_vehicle = vehicles_s[[
    'Num_Acc', 'id_vehicule', 'catv', 'manv', 'obs', 'obsm', 'choc', 'motor'
]].rename(columns={
    'catv': 'vehicle_type', 'manv': 'maneuver', 'obs': 'fixed_obstacle',
    'obsm': 'moving_obstacle', 'choc': 'impact_point', 'motor': 'engine_type'
})

#user dimension
dim_user = users_s[[
    'Num_Acc', 'id_usager', 'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1'
]].rename(columns={
    'catu': 'user_category', 'grav': 'severity', 'sexe': 'sex', 'an_nais': 'birth_year',
    'trajet': 'trip_purpose', 'secu1': 'safety_equipment'
})

#save Gold files
fact_accidents.to_csv('data/gold_fait_accidents.csv', index=False)
dim_location.to_csv(  'data/gold_dim_lieu.csv',       index=False)
dim_vehicle.to_csv(   'data/gold_dim_vehicule.csv',   index=False)
dim_user.to_csv(      'data/gold_dim_usager.csv',     index=False)

print(f'fact_accidents : {len(fact_accidents)} rows')
print(f'dim_location   : {len(dim_location)} rows')
print(f'dim_vehicle    : {len(dim_vehicle)} rows')
print(f'dim_user       : {len(dim_user)} rows')
print('[OK] Gold files saved')

fact_accidents : 51058 rows
dim_location   : 70245 rows
dim_vehicle    : 92678 rows
dim_user       : 125187 rows
[OK] Gold files saved


## Part 2C — Medallion Architecture Diagram

![Medallion Architecture](medallion_drawio.png)

## Justification of design choices

### 1. Dropping near-empty columns
Columns `lartpc` (100% empty), `v2` (91.6% empty) and `occutc` (99% empty) were dropped. A column with almost no data carries no usable information.

### 2. Handling DOM-TOM
The 3,344 accidents whose department code is 971-978 or 986-988 correspond to overseas territories. These are not errors. The split is done on the department code rather than a latitude threshold (less reliable: Saint-Pierre-et-Miquelon, dep 975, has a latitude within the mainland range, and southern Corsica can fall below 41 degrees). These rows were set aside to scope the analysis to mainland France, while keeping them available for a future study.

### 3. Replacing -1 with NaN
In this dataset, -1 means 'not specified'. Leaving it as-is would bias statistical computations. NaN is the standard pandas convention for an unknown value.

### 4. Imputing 'Unknown' on missing text fields
Rather than dropping an entire row because of a missing address, we keep the accident and impute 'Unknown'. The other fields (date, severity, weather) remain usable.

### 5. Enrichment
- `time_of_day`: buckets the accident hour into 4 categories (Morning/Afternoon/Evening/Night).
- `max_severity`: the most severe outcome among all users involved in an accident.

### 6. Choice of a star schema (Gold layer)
Three models were considered:
- **Flat table**: everything in a single table, massive duplication, unusable
- **Snowflake schema**: more normalized but more complex, justified only for very large datasets
- **Star schema**: the option chosen

A star schema fits because an accident involves multiple vehicles and multiple users. It performs well for BI analysis, matches what the assignment suggests ('fact table + dimensions'), and stays simple to maintain. The `Num_Acc` join key links every table.